# Day 17 · PM Take-Home Assignment
## NumPy Implementation Lab
**PG Diploma · AI-ML & Agentic AI Engineering · IIT Gandhinagar**

Topics: Array ops, vectorized math, matrix ops, boolean filtering, broadcasting, genfromtxt, sorting, benchmarking

---

In [ ]:
import numpy as np
import time

print(f'NumPy version: {np.__version__}')

---
## Part A — Sensor Array Analytics Dashboard (40%)

### A1. Generate the Dataset — shape (50, 24, 3)

In [ ]:
rng = np.random.default_rng(seed=1313)

N_SENSORS = 50
N_HOURS   = 24
N_METRICS = 3   # [temperature (°C), humidity (%), battery (%)]

# Generate each metric in its specified range, then stack along axis=-1
temperature = rng.uniform(15, 45, size=(N_SENSORS, N_HOURS))   # 15–45 °C
humidity    = rng.uniform(20, 95, size=(N_SENSORS, N_HOURS))   # 20–95 %
battery     = rng.uniform(10, 100, size=(N_SENSORS, N_HOURS))  # 10–100 %

# Stack → shape (50, 24, 3)  axis order: [sensor, hour, metric]
data = np.stack([temperature, humidity, battery], axis=-1)

TEMP_IDX, HUM_IDX, BAT_IDX = 0, 1, 2   # named metric indices

print(f'Dataset shape : {data.shape}   (sensors × hours × metrics)')
print(f'dtype         : {data.dtype}')
print(f'\nMetric ranges (global min / max):')
for name, idx in [('Temperature', TEMP_IDX), ('Humidity', HUM_IDX), ('Battery', BAT_IDX)]:
    print(f'  {name:<12}: {data[:,:,idx].min():.2f} – {data[:,:,idx].max():.2f}')

### A2. Alert Sensors — Temperature > 40°C OR Humidity > 90%

In [ ]:
# Per (sensor, hour) boolean masks — shape (50, 24)
high_temp = data[:, :, TEMP_IDX] > 40.0
high_hum  = data[:, :, HUM_IDX]  > 90.0

# A sensor is an 'alert sensor' if ANY of its 24 hours triggered either condition
alert_mask    = (high_temp | high_hum).any(axis=1)   # reduce hours → shape (50,)
alert_indices = np.where(alert_mask)[0]

print(f'Alert sensors ({len(alert_indices)} of {N_SENSORS}): {alert_indices.tolist()}')
print(f'\nBreakdown:')
print(f'  Sensors with temp  > 40°C in any hour: {high_temp.any(axis=1).sum()}')
print(f'  Sensors with humid > 90%  in any hour: {high_hum.any(axis=1).sum()}')
print(f'  Union (alert_mask)                    : {alert_mask.sum()}')

### A3. Per-Sensor Daily Averages — result shape (50, 3)

In [ ]:
# Mean over axis=1 (hours) → collapse 24 hours → shape (50, 3)
daily_avg = data.mean(axis=1)

print(f'daily_avg shape : {daily_avg.shape}   (sensors × metrics)')
print(f'\nSample — first 5 sensors:')
print(f'{"Sensor":<9}{"Avg Temp (°C)":<16}{"Avg Humidity (%)":<18}{"Avg Battery (%)"}' )
print('-' * 58)
for i in range(5):
    print(f'{i:<9}{daily_avg[i, TEMP_IDX]:<16.2f}{daily_avg[i, HUM_IDX]:<18.2f}{daily_avg[i, BAT_IDX]:.2f}')

print(f'\nFleet-wide daily averages:')
print(f'  Avg Temperature : {daily_avg[:, TEMP_IDX].mean():.2f} °C')
print(f'  Avg Humidity    : {daily_avg[:, HUM_IDX].mean():.2f} %')
print(f'  Avg Battery     : {daily_avg[:, BAT_IDX].mean():.2f} %')

### A4. Hour with Highest Average Temperature

In [ ]:
# Mean over axis=0 (sensors) → shape (24, 3)  — one average per hour per metric
hourly_avg = data.mean(axis=0)

# Extract temperature column → shape (24,), then argmax
hourly_temp_avg = hourly_avg[:, TEMP_IDX]          # shape (24,)
hottest_hour    = int(np.argmax(hourly_temp_avg))  # scalar

print(f'Hourly average temperatures (°C):')
for h, t in enumerate(hourly_temp_avg):
    marker = ' ← HOTTEST' if h == hottest_hour else ''
    print(f'  Hour {h:02d}: {t:.2f}{marker}')

print(f'\nHottest hour index : {hottest_hour}  ({hottest_hour}:00 — {hottest_hour % 12 or 12} {"AM" if hottest_hour < 12 else "PM"})')
print(f'Peak avg temp      : {hourly_temp_avg[hottest_hour]:.2f} °C')

### A5. Battery Drain Analysis

In [ ]:
# Battery at first hour (index 0) and last hour (index 23) for each sensor
battery_first = data[:, 0,  BAT_IDX]    # shape (50,)
battery_last  = data[:, -1, BAT_IDX]    # shape (50,)

battery_drain = battery_first - battery_last   # positive = drain

critical_mask    = battery_drain > 50.0
critical_sensors = np.where(critical_mask)[0]

print(f'Battery drain per sensor (first − last hour):')
print(f'  Mean drain : {battery_drain.mean():.2f}%')
print(f'  Max drain  : {battery_drain.max():.2f}%  (Sensor {battery_drain.argmax()})')
print(f'  Min drain  : {battery_drain.min():.2f}%  (Sensor {battery_drain.argmin()})')
print(f'\nSensors with critical drain (>50%): {critical_sensors.tolist()}')
print(f'  Count: {len(critical_sensors)} of {N_SENSORS}')
if len(critical_sensors):
    for idx in critical_sensors:
        print(f'  Sensor {idx:02d}: start={battery_first[idx]:.1f}% → end={battery_last[idx]:.1f}%  (drain={battery_drain[idx]:.1f}%)')

### A6. Min-Max Normalize ALL Metrics with Broadcasting

In [ ]:
# Normalise per metric (axis 2 = metric dimension)
# min/max over sensors AND hours → shape (3,)
# We need to keep dims for broadcasting against (50, 24, 3)

metric_min = data.min(axis=(0, 1), keepdims=True)   # shape (1, 1, 3)
metric_max = data.max(axis=(0, 1), keepdims=True)   # shape (1, 1, 3)
metric_rng = metric_max - metric_min                 # shape (1, 1, 3)

# Safe divide (guard constant metrics, though none expected here)
safe_rng = np.where(metric_rng == 0, 1, metric_rng)

# Broadcasting: (50,24,3) - (1,1,3) / (1,1,3)  → (50,24,3)
data_norm = (data - metric_min) / safe_rng

print(f'Normalised data shape : {data_norm.shape}')
print(f'\nGlobal min per metric (should be 0.0):')
print(f'  {data_norm.min(axis=(0,1)).round(6)}')
print(f'Global max per metric (should be 1.0):')
print(f'  {data_norm.max(axis=(0,1)).round(6)}')
print(f'\nSample — Sensor 0, Hours 0-3 (normalised):')
print(f'{"Hour":<7}{"Temp [0-1]":<14}{"Humid [0-1]":<14}{"Batt [0-1]"}')
for h in range(4):
    row = data_norm[0, h]
    print(f'{h:<7}{row[TEMP_IDX]:<14.4f}{row[HUM_IDX]:<14.4f}{row[BAT_IDX]:.4f}')

### A7. Save Daily Averages as `sensor_summary.csv`

In [ ]:
CSV_PATH = 'sensor_summary.csv'

# Save: 50 rows × 3 columns, 4 decimal places, comma-separated
np.savetxt(
    CSV_PATH,
    daily_avg,
    delimiter=',',
    header='avg_temperature_c,avg_humidity_pct,avg_battery_pct',
    comments='',      # suppress the default '#' prefix on the header line
    fmt='%.4f'
)

# ── Verify with genfromtxt ────────────────────────────────────────────────
loaded = np.genfromtxt(CSV_PATH, delimiter=',', skip_header=1)
print(f'Saved  → {CSV_PATH}')
print(f'Shape  : {loaded.shape}')
print(f'Max abs diff from original : {np.abs(loaded - daily_avg).max():.2e}  (rounding only)')
print(f'\nFirst 5 rows of saved CSV:')
print(f'avg_temperature_c  avg_humidity_pct  avg_battery_pct')
for row in loaded[:5]:
    print(f'{row[0]:<19.4f}{row[1]:<18.4f}{row[2]:.4f}')

---
## Part B — Stretch: NumPy Linear Algebra (30%)

### B1. 3×3 Matrix — Determinant, Inverse, Eigenvalues

In [ ]:
A = np.array([[4.0, 2.0, 1.0],
              [3.0, 7.0, 2.0],
              [1.0, 3.0, 5.0]])

det_A      = np.linalg.det(A)
A_inv      = np.linalg.inv(A)
eigvals, eigvecs = np.linalg.eig(A)

print('Matrix A:')
print(A)
print(f'\nDeterminant  : {det_A:.6f}  (≠ 0 → matrix is invertible ✓)')

print(f'\nInverse A⁻¹:')
print(A_inv.round(6))

# Verification: A @ A_inv should equal the 3×3 identity
product = A @ A_inv
identity = np.eye(3)
is_identity = np.allclose(product, identity, atol=1e-10)
print(f'\nA @ A⁻¹  (should be I₃):')
print(product.round(10))
print(f'np.allclose(A @ A_inv, I₃) → {is_identity}  ✓')

print(f'\nEigenvalues  : {eigvals.round(6)}')
print(f'Eigenvectors (columns):')
print(eigvecs.round(6))

# Verify Av = λv for each eigenpair
print('\nVerification  A @ v  ≈  λ × v  for each eigenpair:')
for i in range(3):
    lhs = A @ eigvecs[:, i]
    rhs = eigvals[i] * eigvecs[:, i]
    ok = np.allclose(lhs, rhs)
    print(f'  λ{i} = {eigvals[i]:.4f}  →  {ok} ✓')

### B2. Solve a System of Linear Equations

In [ ]:
# System:  2x + 3y = 8
#          4x +  y = 10
#
# Matrix form:  A @ [x, y]ᵀ = b

A_sys = np.array([[2.0, 3.0],
                  [4.0, 1.0]])
b_sys = np.array([8.0, 10.0])

solution = np.linalg.solve(A_sys, b_sys)
x, y = solution

print('System of equations:')
print('  2x + 3y = 8')
print('  4x +  y = 10')
print(f'\nSolution via np.linalg.solve():')
print(f'  x = {x:.6f}')
print(f'  y = {y:.6f}')

# Verification
residual = np.abs(A_sys @ solution - b_sys)
print(f'\nVerification (A @ x - b should be ≈ 0):')
print(f'  Residual: {residual}  → allclose: {np.allclose(A_sys @ solution, b_sys)} ✓')
print(f'\nManual check:')
print(f'  2({x:.4f}) + 3({y:.4f}) = {2*x + 3*y:.4f}  (expected 8.0)')
print(f'  4({x:.4f}) +  ({y:.4f}) = {4*x + y:.4f}  (expected 10.0)')

### B3. What is `np.linalg.svd()` and Where Is It Used in ML?

**Singular Value Decomposition (SVD)** factorises any matrix **M** (shape m×n) into three matrices: **M = U Σ Vᵀ**, where U (m×m) and V (n×n) are orthogonal matrices containing left and right singular vectors, and Σ (m×n) is a diagonal matrix of non-negative **singular values** sorted in descending order. Each singular value represents the "strength" or variance captured by the corresponding pair of singular vectors.

In ML, SVD is foundational to **Principal Component Analysis (PCA)**: the right singular vectors of the centred data matrix are exactly the principal components, so calling `np.linalg.svd` on the data directly computes PCA without forming the covariance matrix — numerically more stable for wide datasets. SVD is also the engine behind **collaborative filtering recommendation systems** (e.g., Netflix Prize): the user-item ratings matrix is decomposed into latent user and item factor matrices, and low-rank reconstruction predicts missing ratings.

Beyond these two, SVD is used in **latent semantic analysis (LSA)** for document similarity, **pseudo-inverse computation** (`np.linalg.pinv`), and **image compression** — a rank-k truncated SVD approximates the image with only k singular triplets, dramatically reducing storage while retaining most visual information.

In [ ]:
# Quick SVD demo on a 4×3 matrix
M = np.array([[1,2,3],[4,5,6],[7,8,9],[10,11,12]], dtype=float)
U, S, Vt = np.linalg.svd(M, full_matrices=False)

print(f'M shape: {M.shape}')
print(f'U: {U.shape},  S (singular values): {S.round(4)},  Vt: {Vt.shape}')

# Reconstruct M from U, S, Vt
M_reconstructed = U @ np.diag(S) @ Vt
print(f'Reconstruction error: {np.abs(M - M_reconstructed).max():.2e}  (should be ~0) ✓')

# Rank-1 approximation (keep only the largest singular value)
M_rank1 = S[0] * np.outer(U[:, 0], Vt[0, :])
explained = S[0]**2 / (S**2).sum()
print(f'Rank-1 approximation captures {explained*100:.1f}% of total variance')

---
## Part C — Interview Ready (20%)

### C1. Explain the Performance Problem + Vectorized Rewrite

#### Why the Loop Code is Slow

The nested loop operates **element by element in Python**, incurring four sources of overhead for every single value:

1. **Python interpreter overhead** — each `for` iteration, `range()` call, and `result.append()` involves Python bytecode dispatch, object allocation, and reference counting. For a 1000×1000 matrix that's **1 million loop iterations**.
2. **No SIMD / vectorization** — modern CPUs have SIMD registers (AVX-512 etc.) that can process 8–16 floating-point values in a single clock cycle. Python loops cannot exploit these; NumPy's C backend can.
3. **Dynamic `list.append()`** — Python lists reallocate memory repeatedly as they grow, causing cache misses.
4. **Redundant `.reshape()`** — converting a flat list back to an ndarray is an extra O(n) pass.

The expression `x² + 2x + 1` is simply **(x+1)²**, which NumPy evaluates on the entire array in one vectorized C call — all arithmetic is fused into a single memory pass.

**Speedup estimate for 1000×1000:** NumPy vectorized operations typically outperform pure Python loops by **100×–500×** on numerical array work. For this polynomial expression on 10⁶ elements, a realistic benchmark shows ~200× speedup.

In [ ]:
SIZE = 1000
data_bench = np.random.rand(SIZE, SIZE).astype(np.float64)

# ── Slow: nested Python loop ──────────────────────────────────────────────
t0 = time.perf_counter()
result_slow = []
for i in range(len(data_bench)):
    for j in range(len(data_bench[0])):
        result_slow.append(data_bench[i][j] ** 2 + 2 * data_bench[i][j] + 1)
result_slow = np.array(result_slow).reshape(data_bench.shape)
t_slow = time.perf_counter() - t0

# ── Fast: one vectorized line ─────────────────────────────────────────────
t0 = time.perf_counter()
result_fast = (data_bench + 1) ** 2      # algebraically equivalent: x² + 2x + 1 = (x+1)²
t_fast = time.perf_counter() - t0

print(f'Loop (slow)    : {t_slow*1000:.1f} ms')
print(f'Vectorized     : {t_fast*1000:.2f} ms')
print(f'Speedup        : {t_slow/t_fast:.0f}×')
print(f'Results match  : {np.allclose(result_slow, result_fast)} ✓')

### C2. `k_nearest()` — K Nearest Neighbours with NumPy

In [ ]:
def k_nearest(data: np.ndarray, point: np.ndarray, k: int) -> np.ndarray:
    """
    Return indices of the k closest points to `point` in `data`.

    Parameters
    ----------
    data  : (n, 2) array of 2D data points
    point : (2,)  query point
    k     : number of nearest neighbours to return

    Returns
    -------
    indices : (k,) array of integer indices into `data`, sorted nearest-first

    Strategy
    --------
    Subtract point from every row (broadcasting), square, sum across columns,
    sqrt → Euclidean distances. argsort gives indices in ascending distance
    order; take first k.
    """
    diffs     = data - point                          # (n, 2) — broadcasting (n,2)-(2,)
    distances = np.sqrt((diffs ** 2).sum(axis=1))     # (n,)  — Euclidean distance
    return np.argsort(distances)[:k]                  # (k,)  — nearest-first indices


# ── Tests ─────────────────────────────────────────────────────────────────
np.random.seed(99)
dataset = np.random.randn(20, 2)
query   = np.array([0.0, 0.0])
k       = 3

neighbours = k_nearest(dataset, query, k)

print(f'Query point  : {query}')
print(f'k            : {k}')
print(f'\nNearest {k} neighbour indices: {neighbours}')
print(f'Their coordinates:')
for idx in neighbours:
    dist = np.sqrt(((dataset[idx] - query)**2).sum())
    print(f'  Index {idx:3d}: {dataset[idx]}  dist={dist:.4f}')

# Edge case: k = len(data) should return all indices sorted
all_sorted = k_nearest(dataset, query, len(dataset))
print(f'\nAll indices sorted by distance (k=n): {all_sorted}')

# Verify: distances are actually non-decreasing
dists_sorted = np.sqrt(((dataset[all_sorted] - query)**2).sum(axis=1))
print(f'Distances non-decreasing: {np.all(np.diff(dists_sorted) >= 0)} ✓')

### C3. Debug — Column Normalisation Code (3 bugs)

#### Original Buggy Code

```python
data = np.random.randn(100, 5)
means = data.mean(axis=1)      # Line A
stds  = data.std(axis=1)       # Line B
normalized = (data - means) / stds   # Line C
```

---

#### Bug 1 — Line A: `axis=1` should be `axis=0`

**Goal:** Normalise *columns* — each column should have its own mean and std subtracted/divided.  
`axis=1` computes the mean *across columns for each row*, producing shape `(100,)` — one mean per sample. `axis=0` computes the mean *across rows for each column*, producing shape `(5,)` — one mean per feature. The latter is what column standardisation requires.

#### Bug 2 — Line B: `axis=1` should be `axis=0` (same reason)

Same problem as Bug 1. `data.std(axis=1)` gives per-row standard deviations — shape `(100,)`. We need per-column standard deviations — shape `(5,)`, so `axis=0`.

#### Bug 3 — Line C: Broadcasting shape mismatch (from Bugs 1 & 2, AND even if axis was correct)

Even if `means` and `stds` were computed with `axis=1` intentionally, `(data - means)` with shapes `(100, 5) - (100,)` raises a `ValueError` because the trailing dimension `5 ≠ 100`. NumPy aligns from the right: `(100, 5) - (100,)` → trailing dim 5 vs 100 → broadcast fails.

**Fix requires `keepdims=True`** so the result shapes are `(1, 5)` (for `axis=0`) or `(100, 1)` (for `axis=1`), enabling correct broadcasting against `(100, 5)`.

Additionally, columns with `std = 0` (constant feature) cause **division by zero** → `NaN`. This must be guarded with `np.where`.

#### Corrected Code

In [ ]:
np.random.seed(0)
data_c3 = np.random.randn(100, 5)

# Fix 1 & 2: axis=0 (per column); keepdims=True for broadcasting
means = data_c3.mean(axis=0, keepdims=True)    # shape (1, 5)
stds  = data_c3.std(axis=0,  keepdims=True)    # shape (1, 5)

# Fix 3: guard zero-std columns to avoid NaN/inf
safe_stds = np.where(stds == 0, 1, stds)       # shape (1, 5)

# Broadcasting: (100, 5) - (1, 5) / (1, 5)  → (100, 5)
normalized = (data_c3 - means) / safe_stds

print('Verification of corrected normalisation:')
print(f'  Output shape              : {normalized.shape}')
print(f'  Column means (should ≈ 0) : {normalized.mean(axis=0).round(10)}')
print(f'  Column stds  (should ≈ 1) : {normalized.std(axis=0).round(6)}')
print(f'  Any NaN?                  : {np.isnan(normalized).any()}')

# Demonstrate the original failure
print('\nOriginal code failures:')
try:
    bad_means = data_c3.mean(axis=1)       # (100,)
    bad = data_c3 - bad_means              # (100,5) - (100,) → shape conflict
except ValueError as e:
    print(f'  Line C ValueError: {e}')

print(f'  axis=1 means shape: {data_c3.mean(axis=1).shape}  (wrong — gives per-row, not per-column)')
print(f'  axis=0 means shape: {data_c3.mean(axis=0).shape}  (correct — one mean per feature)')

---
## Part D — AI-Augmented Task: IQR Outlier Detection (10%)

### D1. Exact Prompt Used

> *"Write a NumPy function that performs IQR-based outlier detection on each column of a 2D array, replacing outliers with the column median."*

### D2. AI Output (verbatim)

In [ ]:
# ── AI-Generated Function (verbatim) ─────────────────────────────────────
def iqr_outlier_replace(arr):
    """
    Detect and replace outliers in each column using IQR method.
    Outliers are values below Q1 - 1.5*IQR or above Q3 + 1.5*IQR.
    Outliers are replaced with the column median.

    Parameters:
        arr (np.ndarray): 2D input array of shape (n_samples, n_features)

    Returns:
        np.ndarray: Array with outliers replaced by column median
    """
    result = arr.copy()

    for col in range(arr.shape[1]):             # ← loops over columns
        column = arr[:, col]
        Q1 = np.percentile(column, 25)
        Q3 = np.percentile(column, 75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        median = np.median(column)
        outlier_mask = (column < lower) | (column > upper)
        result[outlier_mask, col] = median

    return result

### D3. Critical Evaluation (≤150 words)

**IQR per column:** The AI uses `axis=0` implicitly via the `for col` loop — correct logic, but not truly vectorized. A fully vectorized version would use `np.percentile(..., axis=0)` to compute all Q1/Q3 values at once in one call.

**In-place vs copy:** `result = arr.copy()` is correct — it avoids mutating the input. ✅

**Vectorization:** The function **loops over columns**, violating the vectorization requirement. For 5 columns this is negligible, but for a 10,000-column dataset this matters. The loop is unnecessary — `np.percentile` accepts `axis=0` natively.

**Edge case missed:** A column with **all identical values** (IQR = 0, every value is an "outlier" by the fence formula since lower = upper = the constant) will **replace every value with the median** — which is the same constant — so the result happens to be correct, but for the wrong reason. No explicit guard or warning is issued.

**Test:** Works correctly on known-outlier data (verified below).

In [ ]:
# ── Improved fully-vectorized version ────────────────────────────────────
def iqr_outlier_replace_v2(arr: np.ndarray) -> np.ndarray:
    """
    IQR-based outlier replacement — fully vectorized (no column loop).

    Improvements over AI version:
    - np.percentile with axis=0 computes all columns in one C call
    - np.where for replacement instead of boolean index assignment
    - keepdims=True enables clean broadcasting
    """
    if arr.ndim != 2:
        raise ValueError(f'Expected 2D array, got shape {arr.shape}')

    Q1 = np.percentile(arr, 25, axis=0, keepdims=True)   # (1, n_features)
    Q3 = np.percentile(arr, 75, axis=0, keepdims=True)   # (1, n_features)
    IQR = Q3 - Q1                                         # (1, n_features)

    lower  = Q1 - 1.5 * IQR                              # (1, n_features)
    upper  = Q3 + 1.5 * IQR                              # (1, n_features)
    median = np.median(arr, axis=0, keepdims=True)        # (1, n_features)

    # Broadcasting: (n, f) compared against (1, f) → mask shape (n, f)
    outlier_mask = (arr < lower) | (arr > upper)

    # np.where: where mask=True use median (broadcast), else keep original
    return np.where(outlier_mask, median, arr)            # returns a copy ✓


# ── Test with known outliers ───────────────────────────────────────────────
rng_test = np.random.default_rng(77)
test_data = rng_test.normal(0, 1, (100, 4))

# Inject obvious outliers
test_data[0, 0]  = 999.0    # extreme outlier in col 0
test_data[50, 2] = -500.0   # extreme outlier in col 2
test_data[99, 3] = 200.0    # extreme outlier in col 3

out_ai = iqr_outlier_replace(test_data.copy())
out_v2 = iqr_outlier_replace_v2(test_data)

print('Known outlier tests:')
print(f'  Col 0, row 0  — original: {test_data[0,0]:8.1f}  AI: {out_ai[0,0]:.4f}  V2: {out_v2[0,0]:.4f}')
print(f'  Col 2, row 50 — original: {test_data[50,2]:8.1f}  AI: {out_ai[50,2]:.4f}  V2: {out_v2[50,2]:.4f}')
print(f'  Col 3, row 99 — original: {test_data[99,3]:8.1f}  AI: {out_ai[99,3]:.4f}  V2: {out_v2[99,3]:.4f}')
print(f'\nResults match: {np.allclose(out_ai, out_v2)} ✓')

# Timing comparison
big = rng_test.normal(0, 1, (10000, 500))
t0 = time.perf_counter(); iqr_outlier_replace(big.copy()); t_ai = time.perf_counter() - t0
t0 = time.perf_counter(); iqr_outlier_replace_v2(big);     t_v2 = time.perf_counter() - t0
print(f'\nPerformance on (10000, 500) array:')
print(f'  AI (loop)   : {t_ai*1000:.1f} ms')
print(f'  V2 (vector) : {t_v2*1000:.1f} ms')
print(f'  Speedup     : {t_ai/t_v2:.1f}×')